# Histogram gradient boosting: training and results

The runner uses four arms, ten patient-grouped outer folds and three inner folds.
Depth is selected by inner macro-F1, with one boundary refinement. The supplied HGB report
records completed scores, but its fold predictions and tuning files are
missing from this checkout; those scores cannot currently be verified.

Inspection is the default. Set `HGB_RUN_TRAINING=1` before opening Jupyter to run
the experiment into a new reproduction directory. This may take substantial time.

Before training, install dependencies with `python -m pip install -r models/histogram_gradient_boosting/requirements.txt`. The historical environment was not recorded; the current dependency pin is for future runs.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys
import pandas as pd
from IPython.display import display

PROJECT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                if (p / 'models/histogram_gradient_boosting/run_experiment.py').is_file()
                and (p / 'src/afib').is_dir()), None)
if PROJECT is None:
    raise FileNotFoundError('Run from the repository or one of its subdirectories.')
RUN_TRAINING = os.environ.get('HGB_RUN_TRAINING', '0') == '1'
REFRESH_REPORTS = RUN_TRAINING or os.environ.get('HGB_REFRESH_REPORTS', '0') == '1'
OUTPUT = PROJECT / ('results/histogram_gradient_boosting/reproduction' if RUN_TRAINING
                    else 'results/histogram_gradient_boosting/formal')
print('Training enabled:', RUN_TRAINING)
print('Results:', OUTPUT)

In [ ]:
if RUN_TRAINING:
    subprocess.run([sys.executable, str(PROJECT / 'models/histogram_gradient_boosting/run_experiment.py'),
                    '--arms', 'A', 'B', 'C', 'D', '--outer-folds', *map(str, range(10)),
                    '--output', str(OUTPUT)], cwd=PROJECT, check=True)
else:
    print('Inspection mode: no models fitted.')

In [ ]:
required = [OUTPUT / f'{arm}_fold{fold}' / name for arm in 'ABCD' for fold in range(10)
            for name in ['complete.json', 'inner_tuning.csv', 'metrics.csv', 'predictions.csv']]
missing = [path for path in required if not path.is_file()]
READY = not missing
if READY:
    if REFRESH_REPORTS:
        subprocess.run([sys.executable, str(PROJECT / 'models/histogram_gradient_boosting/check_results.py'),
                        '--output', str(OUTPUT)], cwd=PROJECT, check=True)
    metrics = pd.concat([pd.read_csv(OUTPUT / f'{arm}_fold{fold}/metrics.csv')
                         for arm in 'ABCD' for fold in range(10)], ignore_index=True)
    observed = metrics.loc[metrics.scenario.eq('observed') & metrics.repeat.eq(0)]
    display(observed.groupby('arm')[['macro_f1', 'sensitivity', 'specificity', 'brier']].agg(['mean', 'std']))
    tuning = pd.concat([pd.read_csv(OUTPUT / f'{arm}_fold{fold}/inner_tuning.csv')
                        for arm in 'ABCD' for fold in range(10)], ignore_index=True)
    display(tuning.groupby(['arm', 'max_depth']).macro_f1.agg(['count', 'mean', 'std']))
    selected = [json.loads((OUTPUT / f'{arm}_fold{fold}/complete.json').read_text())
                for arm in 'ABCD' for fold in range(10)]
    display(pd.DataFrame(selected)[['arm', 'outer_fold', 'best_depth']])
else:
    print(f'Results unavailable or incomplete: {len(missing)} of {len(required)} required files are missing.')

The checker validates fold/scenario coverage and prediction structure and rebuilds
summary tables. It does not establish historical source hashes or independently
recompute every metric. Use `docs/Histogram_Gradient_Boosting_Observations.md` for
reported scores and evidence limits. Outer-fold sample SD is descriptive variation,
not a confidence interval. No absent tuning choices are inferred from aggregate scores.